# N°37 — Pandas 2, Clase 1 (A+C): limpiar a número + descartar filas sucias

**Objetivo de la sesión:** Limpiar una columna numérica sucia con `pd.to_numeric()` y descartar las filas que no se pudieron limpiar. Es la técnica con la sintaxis más densa del bloque de pandas — se le da la sesión completa, sin compartirla con nada más.

## ✅ Contenido que debe quedar consolidado al cierre

- Limpiar a número: `pd.to_numeric(serie.str.replace(...), errors="coerce")` (sin función propia, sin `try/except`)
- Descartar filas sucias **solo por esta columna** — `df[df["col"].notna()]`, misma técnica de filtrar filas que ya vieron en N°36 con `sexo.notna()`

**No se toca todavía** la columna de categorías (`sexo`) — eso es N°38. El descarte combinado de las dos limpiezas juntas se hace recién al abrir N°39, como puente.

### 💻 Lo que se espera que un estudiante escriba (ejemplo mínimo, con `guaguas`)

In [ ]:
import pandas as pd

tabla = pd.read_csv("guaguas_maquillada.csv", dtype={"n": "str"})

tabla["cantidad_guaguas"] = pd.to_numeric(
    tabla["n"].astype(str).str.strip().str.replace(".", "", regex=False), errors="coerce"
)
tabla[["n", "cantidad_guaguas"]].head()

### ⚠️ Por qué `dtype={"n": "str"}` es obligatorio, no opcional

Al leer `guaguas_maquillada.csv` **sin** este parámetro, pandas adivina el tipo de la columna `n` mirando todos sus valores. El problema: un valor sucio con punto de miles (`"2.908"`) *también* es un número válido para pandas — solo que semanticamente incorrecto (2,908 en vez de 2908). Pandas lo convierte a `float64` **sin avisar, sin error, sin `NaN`** — la fila queda con un dato equivocado para siempre, y `pd.to_numeric(..., errors="coerce")` ya no tiene nada que limpiar.

`dtype={"n": "str"}` le dice a pandas que no adivine esa columna: la deja como texto, tal cual está escrita en el archivo, para que la limpiemos nosotros con la técnica de arriba. Con este fix se recupera el 100% de los valores sucios-pero-válidos (verificado contra el archivo real: 4.293 nulos antes de leer, 4.293 nulos después de limpiar — cero pérdida de datos).

**Verificado en esta sesión (2026-09-24):** sin el fix, `tabla["n"].dtype` da `float64` y la fila 51726 (Luis, 1933) se lee como `2.908` en vez de `2908`. Con el fix, se recupera correcto.

**Nota para clases futuras del bloque:** N°39 ("derivar, agrupar, ordenar") reabre `guaguas.csv` y reaplica esta limpieza como puente — su lectura de `n` también necesita `dtype={"n": "str"}`, si no, reaparece el mismo problema. N°38 no toca `n`, así que no le aplica.

In [ ]:
tabla_sin_nulos_numero = tabla[tabla["cantidad_guaguas"].notna()]
print("Filas antes:", len(tabla))
print("Filas después de descartar cantidad_guaguas nula:", len(tabla_sin_nulos_numero))

## 📝 Nota

⚠️ Pandas no entra en la Evaluación de Funciones+Strings+Listas (fecha pendiente) — decirlo explícito para no generar ansiedad.